## Conectando Python ao PostgreSQL

In [1]:
# Biblioteca para conectar a PostgreSQL
%pip install "psycopg[binary]" -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psycopg
import os

## Criando a conexão

In [3]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="lh_nautical",
    user="postgres",
    password="Admin123"
)

In [4]:
# Testar a conexão com o banco de dados
with conn.cursor() as cursor:
    cursor.execute("SELECT 1;")
    resultado = cursor.fetchone()

print(resultado)

(1,)


In [5]:
# Confirmar o banco de dados e o schema atual
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT current_database(), current_schema();
    """)
    
    print(cursor.fetchone())

('lh_nautical', 'public')


## Criando schema no PostgreSQL

In [6]:
# Executar o script SQL para criar o schema
with open("schema_final.sql", "r", encoding="utf-8") as arquivo:
    schema_sql = arquivo.read()

with conn.cursor() as cursor:
    cursor.execute(schema_sql)

conn.commit()
print("Schema criado com sucesso!")

Schema criado com sucesso!


In [7]:
# Verificar se todas as tabelas foram criadas corretamente
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_type = 'BASE TABLE'
        ORDER BY table_name;
    """)
    tabelas = cursor.fetchall()

print(f"Quantidade de tabelas: {len(tabelas)}")

for tabela in tabelas:
    print(tabela[0])

Quantidade de tabelas: 24
addresses
attributes
brands
categories
customers
employees
fiscal_invoices
goods_receipt_items
goods_receipts
locations
order_items
orders
payments
product_suppliers
product_variants
products
purchase_order_items
purchase_orders
return_items
returns
stock_levels
stock_movements
suppliers
variant_attribute_values


In [ ]:
# Especifica a pasta onde estão os arquivos CSV
pasta = "1-lh_nautical_csv"
arquivos = os.listdir(pasta) 

In [9]:
arquivos_csv = [
    arquivo
    for arquivo in os.listdir(pasta)
    if arquivo.lower().endswith(".csv")
]

print(f"CSV encontrados: {len(arquivos_csv)}")
arquivos_csv

CSV encontrados: 24


['addresses.csv',
 'attributes.csv',
 'brands.csv',
 'categories.csv',
 'customers.csv',
 'employees.csv',
 'fiscal_invoices.csv',
 'goods_receipts.csv',
 'goods_receipt_items.csv',
 'locations.csv',
 'orders.csv',
 'order_items.csv',
 'payments.csv',
 'products.csv',
 'product_suppliers.csv',
 'product_variants.csv',
 'purchase_orders.csv',
 'purchase_order_items.csv',
 'returns.csv',
 'return_items.csv',
 'stock_levels.csv',
 'stock_movements.csv',
 'suppliers.csv',
 'variant_attribute_values.csv']

In [ ]:
"""
# Carregar o arquivo específico "orders.csv" para teste
arquivo = os.path.join(pasta, "orders.csv")

with conn.cursor() as cursor:
    with cursor.copy(
        'COPY "orders" FROM STDIN WITH (FORMAT CSV, HEADER TRUE)'
    ) as copy:

        with open(
            arquivo,
            "rb"
        ) as csv_file:

            while dados := csv_file.read(8192):
                copy.write(dados)

conn.commit()
"""

## Carregando os 24 arquivos

In [10]:
# Carregar todos os arquivos CSV da pasta
def carregar_csv(conn, caminho, nome_tabela):

    with conn.cursor() as cursor:

        comando = f'''
        COPY "{nome_tabela}"
        FROM STDIN
        WITH (FORMAT CSV, HEADER TRUE)
        '''

        with cursor.copy(comando) as copy:

            with open(caminho, "rb") as arquivo:
                while dados := arquivo.read(8192): # Lê o arquivo em blocos de 8192 bytes
                    copy.write(dados)

    conn.commit()

In [11]:
for arquivo_csv in arquivos_csv:

    caminho = os.path.join(pasta, arquivo_csv)
    nome_tabela = os.path.splitext(arquivo_csv)[0]

    print(f"Carregando: {arquivo_csv}")

    carregar_csv(
        conn,
        caminho,
        nome_tabela
    )

print("Carregamento concluído!")

Carregando: addresses.csv
Carregando: attributes.csv
Carregando: brands.csv
Carregando: categories.csv
Carregando: customers.csv
Carregando: employees.csv
Carregando: fiscal_invoices.csv
Carregando: goods_receipts.csv
Carregando: goods_receipt_items.csv
Carregando: locations.csv
Carregando: orders.csv
Carregando: order_items.csv
Carregando: payments.csv
Carregando: products.csv
Carregando: product_suppliers.csv
Carregando: product_variants.csv
Carregando: purchase_orders.csv
Carregando: purchase_order_items.csv
Carregando: returns.csv
Carregando: return_items.csv
Carregando: stock_levels.csv
Carregando: stock_movements.csv
Carregando: suppliers.csv
Carregando: variant_attribute_values.csv
Carregamento concluído!
